# Model Training - Exploit-Aware Vulnerability Prioritization
**Cost-Sensitive, Exploit-Aware Vulnerability Prioritization (IS353IA)**  
Madhur Rishi Sikarwar (1RV24IS067) - Mayur M Deekshith (1RV24IS069)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve
from sklearn.tree import DecisionTreeClassifier, export_text

SEED = 42
CFG = dict(
    n_cves=15000, start="2019-01-01", end="2024-12-31", download_date="2026-01-01",
    target_pos_rate=0.03, label_window_days=365,
    train_end="2021-12-31", val_end="2022-12-31",      # test = everything after val_end
    cost_fn=50, cost_fp=1,                              # a missed exploit is 50x worse than a false alarm
    tfidf_terms=100, top_cwe=20, rf_trees=200, rf_min_leaf=5, dt_depth=6,
)
print("sklearn / pandas ready; seed =", SEED)

In [ ]:
# (product, vendor, type, popularity)
PRODUCTS = [
    ("windows_server", "microsoft", "os", 9), ("windows_10", "microsoft", "os", 9),
    ("ubuntu_linux", "canonical", "os", 6), ("rhel", "redhat", "os", 5),
    ("macos", "apple", "os", 3), ("apache_http", "apache", "server", 6),
    ("nginx", "f5", "server", 5), ("iis", "microsoft", "server", 5),
    ("tomcat", "apache", "server", 4), ("exchange_server", "microsoft", "server", 5),
    ("mysql", "oracle", "database", 4), ("postgresql", "postgresql", "database", 3),
    ("mssql", "microsoft", "database", 4), ("oracle_db", "oracle", "database", 3),
    ("mongodb", "mongodb", "database", 2), ("chrome", "google", "browser", 8),
    ("firefox", "mozilla", "browser", 5), ("edge", "microsoft", "browser", 5),
    ("openssl", "openssl", "library", 6), ("log4j", "apache", "library", 4),
    ("zlib", "zlib", "library", 2), ("curl", "haxx", "library", 3),
    ("glibc", "gnu", "library", 3), ("fortios", "fortinet", "network", 5),
    ("cisco_ios", "cisco", "network", 6), ("pan_os", "paloalto", "network", 3),
    ("junos", "juniper", "network", 3), ("vmware_esxi", "vmware", "app", 4),
    ("wordpress", "automattic", "app", 6), ("jenkins", "jenkins", "app", 4),
    ("confluence", "atlassian", "app", 3), ("gitlab", "gitlab", "app", 3),
    ("adobe_reader", "adobe", "app", 4), ("office", "microsoft", "app", 6),
    ("sharepoint", "microsoft", "app", 3), ("citrix_adc", "citrix", "network", 3),
    ("zimbra", "synacor", "app", 2), ("struts", "apache", "library", 2),
    ("php", "php", "server", 4), ("nodejs", "nodejs", "server", 3),
]
CWES = [79, 89, 20, 22, 78, 787, 125, 416, 502, 94, 287, 200, 352, 434, 476, 119,
        798, 862, 863, 400, 611, 918, 190, 77, 269, 306, 732, 601, 59, 295, 310, 312]
HOT_CWE = {787: 0.8, 502: 1.0, 94: 0.9, 78: 0.8, 416: 0.6, 287: 0.5, 22: 0.4, 89: 0.4, 306: 0.6}
PHRASES = {"remote code execution": 1.3, "authentication bypass": 0.8, "privilege escalation": 0.5,
           "SQL injection": 0.4, "buffer overflow": 0.5, "path traversal": 0.3, "arbitrary file upload": 0.6,
           "deserialization of untrusted data": 0.9, "cross-site scripting": -0.4, "denial of service": -0.6,
           "information disclosure": -0.4, "use after free": 0.4}
VECTORS = ["network request", "crafted packet", "malicious document", "HTTP request",
           "local user session", "crafted file", "web interface", "API call"]


def cvss31_base(av, ac, pr, ui, s, c, i, a):
    """Real CVSS v3.1 base-score formula."""
    AV = pd.Series(av).map({"N": .85, "A": .62, "L": .55, "P": .2}).to_numpy()
    AC = pd.Series(ac).map({"L": .77, "H": .44}).to_numpy()
    ch = np.asarray(s) == "C"
    PR = np.where(ch, pd.Series(pr).map({"N": .85, "L": .68, "H": .5}).to_numpy(),
                  pd.Series(pr).map({"N": .85, "L": .62, "H": .27}).to_numpy())
    UI = pd.Series(ui).map({"N": .85, "R": .62}).to_numpy()
    m = {"H": .56, "L": .22, "N": 0.0}
    C, I, A = (pd.Series(x).map(m).to_numpy() for x in (c, i, a))
    iss = 1 - (1 - C) * (1 - I) * (1 - A)
    imp = np.where(ch, 7.52 * (iss - .029) - 3.25 * (iss - .02) ** 15, 6.42 * iss)
    ex = 8.22 * AV * AC * PR * UI
    base = np.where(imp <= 0, 0, np.where(ch, np.minimum(1.08 * (imp + ex), 10), np.minimum(imp + ex, 10)))
    return np.ceil(base * 10) / 10


sigmoid = lambda x: 1 / (1 + np.exp(-x))


def generate(cfg, seed=SEED):
    rng = np.random.default_rng(seed)
    n = cfg["n_cves"]
    prod = pd.DataFrame(PRODUCTS, columns=["product", "vendor", "ptype", "pop"])
    pw = prod["pop"].to_numpy(float)
    p = prod.iloc[rng.choice(len(prod), n, p=pw / pw.sum())].reset_index(drop=True)
    days = (pd.Timestamp(cfg["end"]) - pd.Timestamp(cfg["start"])).days
    pub = pd.Timestamp(cfg["start"]) + pd.to_timedelta(rng.integers(0, days, n), unit="D")

    netish = p["ptype"].isin(["server", "network", "app", "browser"]).to_numpy()
    av = np.where(rng.random(n) < np.where(netish, .75, .45), "N", rng.choice(["A", "L", "P"], n, p=[.1, .8, .1]))
    ac = rng.choice(["L", "H"], n, p=[.8, .2]);  pr = rng.choice(["N", "L", "H"], n, p=[.55, .35, .10])
    ui = rng.choice(["N", "R"], n, p=[.65, .35]);  s = rng.choice(["U", "C"], n, p=[.8, .2])
    c, i, a = (rng.choice(["H", "L", "N"], n, p=[.4, .35, .25]) for _ in range(3))
    cvss = cvss31_base(av, ac, pr, ui, s, c, i, a)
    w = np.linspace(2, 1, len(CWES))
    cwe = rng.choice(CWES, n, p=w / w.sum())
    phr = rng.choice(list(PHRASES), n);  vec = rng.choice(VECTORS, n)
    n_refs = rng.poisson(4, n) + 1
    desc = [f"{pn.replace('_', ' ')} before {rng.integers(1, 30)}.{rng.integers(0, 20)} allows attackers to cause {ph} via a {v}"
            for pn, ph, v in zip(p["product"], phr, vec)]

    # hidden latent exploitability: what the model has to discover from observable features
    z = (1.2 * (av == "N") + 0.7 * (ac == "L") + 0.7 * (pr == "N") + 0.5 * (ui == "N")
         + 0.5 * ((c == "H").astype(int) + (i == "H") + (a == "H")) / 3 * 3 / 3
         + np.array([HOT_CWE.get(x, 0.0) for x in cwe]) + np.array([PHRASES[k] for k in phr])
         + 0.12 * p["pop"].to_numpy() + 0.1 * np.minimum(n_refs, 8) + rng.normal(0, 0.6, n)) * 2.6
    lo, hi = -80, 20
    for _ in range(60):                       # tune intercept to reach the target positive rate
        mid = (lo + hi) / 2
        lo, hi = (lo, mid) if sigmoid(z + mid).mean() > cfg["target_pos_rate"] else (mid, hi)
    p_true = sigmoid(z + (lo + hi) / 2)
    exploited = rng.random(n) < p_true
    kev_date = pd.Series(pub + pd.to_timedelta(rng.integers(5, 600, n), unit="D")).where(exploited)
    epss = sigmoid(np.log(p_true / (1 - p_true)) + rng.normal(0, 1.0, n))   # richer-signal baseline

    df = pd.DataFrame({"cve_id": [f"CVE-{y}-{k:05d}" for y, k in zip(pub.year, rng.permutation(n) + 1000)],
                       "published": pub, "product": p["product"], "vendor": p["vendor"], "ptype": p["ptype"],
                       "av": av, "ac": ac, "pr": pr, "ui": ui, "scope": s, "c": c, "i": i, "a": a,
                       "cvss": cvss, "cwe": cwe, "n_refs": n_refs, "description": desc, "kev_date": kev_date, "epss": epss})
    # LABEL: 1 if added to KEV within 365 days of publication
    df["label"] = ((df["kev_date"] - df["published"]).dt.days <= cfg["label_window_days"]).fillna(False).astype(int)
    # CENSORING: keep only CVEs that had a full 12 months to be added to KEV before the data download date
    df = df[df["published"] <= pd.Timestamp(cfg["download_date"]) - pd.Timedelta(days=365)]
    return df.sort_values("published").reset_index(drop=True)


df = generate(CFG)
print(f"{len(df):,} CVEs · positive rate {df.label.mean():.2%}")
df.head(3)

In [ ]:
tr = df["published"] <= pd.Timestamp(CFG["train_end"])
va = (df["published"] > pd.Timestamp(CFG["train_end"])) & (df["published"] <= pd.Timestamp(CFG["val_end"]))
te = df["published"] > pd.Timestamp(CFG["val_end"])
summary = pd.DataFrame({"CVEs": [tr.sum(), va.sum(), te.sum()],
                        "exploited": [df.label[tr].sum(), df.label[va].sum(), df.label[te].sum()]},
                       index=["train (fit models)", "validation (calibrate + threshold)", "test (final evaluation)"])
summary["rate %"] = (100 * summary["exploited"] / summary["CVEs"]).round(2)
summary

In [ ]:
CAT = ["av", "ac", "pr", "ui", "scope", "c", "i", "a"]


class Featurizer:
    def fit(self, d):
        self.top_cwe = d["cwe"].value_counts().head(CFG["top_cwe"]).index.tolist()
        self.vendor_f = d["vendor"].value_counts(normalize=True).to_dict()
        self.prod_f = d["product"].value_counts(normalize=True).to_dict()
        self.tfidf = TfidfVectorizer(max_features=CFG["tfidf_terms"], stop_words="english").fit(d["description"])
        self.cols = list(self.transform(d).columns)
        return self

    def transform(self, d):
        parts = [pd.get_dummies(d[CAT].astype(str), dtype=float).reset_index(drop=True)]
        extra = {"cvss": d["cvss"].to_numpy(), "vendor_freq": d["vendor"].map(self.vendor_f).fillna(0).to_numpy(),
                 "product_freq": d["product"].map(self.prod_f).fillna(0).to_numpy(), "n_refs": d["n_refs"].to_numpy()}
        for w in self.top_cwe:
            extra[f"cwe_{w}"] = (d["cwe"] == w).astype(float).to_numpy()
        T = self.tfidf.transform(d["description"]).toarray()
        for k, name in enumerate(self.tfidf.get_feature_names_out()):
            extra[f"tf_{name}"] = T[:, k]
        X = pd.concat(parts + [pd.DataFrame(extra)], axis=1)
        return X.reindex(columns=self.cols, fill_value=0.0) if hasattr(self, "cols") else X


fz = Featurizer().fit(df[tr])
X, y = fz.transform(df), df["label"].to_numpy()
Xtr, Xva, Xte = X[tr.to_numpy()], X[va.to_numpy()], X[te.to_numpy()]
ytr, yva, yte = y[tr], y[va], y[te]
print("feature matrix:", X.shape)

In [ ]:
models = {
    "logreg": LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5),
    "decision_tree": DecisionTreeClassifier(max_depth=CFG["dt_depth"], min_samples_leaf=20, class_weight="balanced", random_state=SEED),
    "random_forest": RandomForestClassifier(n_estimators=CFG["rf_trees"], min_samples_leaf=CFG["rf_min_leaf"],
                                            class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
    "gradient_boosting": HistGradientBoostingClassifier(max_depth=4, learning_rate=0.06, max_iter=200,
                                                        class_weight="balanced", random_state=SEED),
}
mu, sd = Xtr.mean(), Xtr.std().replace(0, 1)          # logistic regression needs scaled inputs
raw = {}
for name, m in models.items():
    if name == "logreg":
        m.fit((Xtr - mu) / sd, ytr);  raw[name] = m.predict_proba((X - mu) / sd)[:, 1]
    else:
        m.fit(Xtr, ytr);  raw[name] = m.predict_proba(X)[:, 1]
    print(f"trained {name}")

In [ ]:
cal, iso = {}, {}
for name in models:
    iso[name] = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(raw[name][va.to_numpy()], yva)
    cal[name] = np.clip(iso[name].predict(raw[name]), 1e-5, 1)
brier = pd.DataFrame({"Brier raw": {n: brier_score_loss(yte, np.clip(raw[n][te.to_numpy()], 0, 1)) for n in models},
                      "Brier calibrated": {n: brier_score_loss(yte, cal[n][te.to_numpy()]) for n in models}}).round(4)
print("Brier score on the TEST period (lower = better probabilities):")
brier

In [ ]:
def best_threshold(y_true, p, c_fn, c_fp):
    ts = np.unique(np.quantile(p, np.linspace(0, 1, 400)))
    cost = [c_fn * ((p < t) & (y_true == 1)).sum() + c_fp * ((p >= t) & (y_true == 0)).sum() for t in ts]
    return float(ts[int(np.argmin(cost))])


def expected_cost(y_true, p, t, c_fn, c_fp):
    return float(c_fn * ((p < t) & (y_true == 1)).sum() + c_fp * ((p >= t) & (y_true == 0)).sum())


rf_cal = cal["random_forest"]
rows = []
for ratio in (10, 25, 50, 100, 200):
    t = best_threshold(yva, rf_cal[va.to_numpy()], ratio, 1)
    p_te = rf_cal[te.to_numpy()]
    rows.append({"FN:FP cost ratio": f"{ratio}:1", "threshold": round(t, 4), "flagged in test": int((p_te >= t).sum()),
                 "caught (TP)": int(((p_te >= t) & (yte == 1)).sum()), "missed (FN)": int(((p_te < t) & (yte == 1)).sum()),
                 "test expected cost": expected_cost(yte, p_te, t, ratio, 1)})
pd.DataFrame(rows)

In [ ]:
def recall_at_k(y_true, score, k=100):
    idx = np.argsort(-np.asarray(score), kind="stable")[:k]
    return float(np.asarray(y_true)[idx].sum() / max(np.asarray(y_true).sum(), 1))


tm = te.to_numpy()
scores = {"Random forest": raw["random_forest"][tm], "Gradient boosting": raw["gradient_boosting"][tm],
          "Logistic regression": raw["logreg"][tm], "Decision tree": raw["decision_tree"][tm],
          "CVSS only (baseline)": df["cvss"].to_numpy()[tm], "EPSS stand-in (baseline)": df["epss"].to_numpy()[tm]}
results = pd.DataFrame({"PR-AUC": {k: average_precision_score(yte, v) for k, v in scores.items()},
                        "Recall@100": {k: recall_at_k(yte, v) for k, v in scores.items()}}).round(3)
print(f"test period: {tm.sum():,} CVEs, {yte.sum()} later exploited (base rate {yte.mean():.2%}; random PR-AUC ~ {yte.mean():.3f})")
results.sort_values("PR-AUC", ascending=False)

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
    for k in ("Random forest", "CVSS only (baseline)", "EPSS stand-in (baseline)"):
        pr_, rc_, _ = precision_recall_curve(yte, scores[k]);  ax[0].plot(rc_, pr_, label=k)
    ax[0].axhline(yte.mean(), ls="--", c="gray");  ax[0].set(title="Precision–recall (test)", xlabel="recall", ylabel="precision");  ax[0].legend()
    ax[1].barh(results.index[::-1], results["PR-AUC"][::-1], color="#2bb673");  ax[1].set(title="PR-AUC (higher is better)")
    pt = rf_cal[tm];  q = pd.qcut(pt, 10, duplicates="drop")
    rel = pd.DataFrame({"p": pt, "y": yte}).groupby(q, observed=True).mean()
    ax[2].plot([0, rel.p.max()], [0, rel.p.max()], "--", c="gray");  ax[2].plot(rel.p, rel.y, "o-", c="#2bb673")
    ax[2].set(title="Calibration of the random forest (test)", xlabel="mean predicted probability", ylabel="observed rate")
    plt.tight_layout();  plt.show()
except ImportError:
    print("matplotlib not installed – skipping plots")

In [ ]:
imp = pd.Series(models["random_forest"].feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 12 random-forest features:\n", imp.head(12).round(4).to_string())
print("\nDecision tree (top 3 levels):\n")
print(export_text(models["decision_tree"], feature_names=list(X.columns), max_depth=3))

In [ ]:
scored = df[["cve_id", "published", "product", "cvss", "epss", "label"]].copy()
scored["p_exploit"] = cal["random_forest"]
scored["split"] = np.where(tr, "train", np.where(va, "val", "test"))
print(scored[scored.split == "test"].sort_values("p_exploit", ascending=False).head(10).round(3).to_string(index=False))
scored.to_csv("scored_cves.csv", index=False)
try:
    from google.colab import files          # only exists inside Colab
    files.download("scored_cves.csv")
except ImportError:
    print("saved scored_cves.csv (not in Colab, so no auto-download)")

In [ ]:
FEATURE_GROUPS = {
    "CVSS vector (AV/AC/PR/UI/S)": [c for c in X.columns if c.split("_")[0] in ("av", "ac", "pr", "ui", "scope")],
    "Impact (C/I/A)": [c for c in X.columns if c.split("_")[0] in ("c", "i", "a")],
    "CVSS base score": ["cvss"],
    "Weakness type (CWE)": [c for c in X.columns if c.startswith("cwe_")],
    "Vendor / product popularity": ["vendor_freq", "product_freq"],
    "Reference count": ["n_refs"],
    "Description wording": [c for c in X.columns if c.startswith("tf_")],
}
rf, rf_iso = models["random_forest"], iso["random_forest"]
BG = Xtr.sample(80, random_state=SEED).reset_index(drop=True)       # 80 real training CVEs as the "typical CVE" background
base_raw = float(rf.predict_proba(BG)[:, 1].mean())
THRESH = best_threshold(yva, cal["random_forest"][va.to_numpy()], CFG["cost_fn"], CFG["cost_fp"])
all_cal = cal["random_forest"]
VENDORS, TYPES = dict(zip(df["product"], df["vendor"])), dict(zip(df["product"], df["ptype"]))


def cvss_parts(av, ac, pr, ui, s, c, i, a):
    AV = {"N": .85, "A": .62, "L": .55, "P": .2}[av];  AC = {"L": .77, "H": .44}[ac]
    PR = ({"N": .85, "L": .68, "H": .5} if s == "C" else {"N": .85, "L": .62, "H": .27})[pr]
    UI = {"N": .85, "R": .62}[ui];  m = {"H": .56, "L": .22, "N": 0.0}
    iss = 1 - (1 - m[c]) * (1 - m[i]) * (1 - m[a])
    imp = 7.52 * (iss - .029) - 3.25 * (iss - .02) ** 15 if s == "C" else 6.42 * iss
    return AV, AC, PR, UI, iss, imp, 8.22 * AV * AC * PR * UI


def explain_cve(cve=None, **custom):
    """Score one CVE and show, step by step, how the score is calculated."""
    known = isinstance(cve, str)
    if known:
        row = df[df["cve_id"] == cve]
        if row.empty:
            print(f"{cve} is not in the dataset - pick an id from the list above, or type NEW to describe your own CVE.");  return
        row = row.iloc[[0]]
    else:
        f = dict(product="apache_http", av="N", ac="L", pr="N", ui="N", scope="U", c="H", i="H", a="H", cwe=787, n_refs=5,
                 description="allows attackers to cause remote code execution via a crafted packet")
        f.update(custom)
        f.setdefault("vendor", VENDORS.get(f["product"], "unknown"));  f.setdefault("ptype", TYPES.get(f["product"], "app"))
        f["cvss"] = float(cvss31_base([f["av"]], [f["ac"]], [f["pr"]], [f["ui"]], [f["scope"]], [f["c"]], [f["i"]], [f["a"]])[0])
        f.setdefault("epss", np.nan);  f["cve_id"] = custom.get("cve_id", "(new CVE)");  f["published"] = pd.Timestamp("2024-06-01")
        row = pd.DataFrame([f])
    r = row.iloc[0]
    x = fz.transform(row)
    tree_p = np.array([t.predict_proba(x.to_numpy())[0, 1] for t in rf.estimators_])      # one probability per tree
    p_raw = float(tree_p.mean())
    p = float(rf_iso.predict([p_raw])[0])
    line = "-" * 78

    print("=" * 78)
    print(f"{r['cve_id']}   |   {r['product']} ({r['vendor']})   |   published {pd.Timestamp(r['published']).date()}")
    print(f"vector: AV:{r['av']} AC:{r['ac']} PR:{r['pr']} UI:{r['ui']} S:{r['scope']} C:{r['c']} I:{r['i']} A:{r['a']}   CWE-{int(r['cwe'])}   refs: {int(r['n_refs'])}")
    print(f"description: {r['description']}")
    print("=" * 78)
    print(f"\n  >>> MODEL EXPLOIT PROBABILITY:  {p:.1%}      (CVSS {r['cvss']:.1f}/10" +
          (f", EPSS baseline {r['epss']:.3f})" if pd.notna(r['epss']) else ")"))
    print(f"  >>> DECISION: {'PATCH CANDIDATE (flagged)' if p >= THRESH else 'below the patch threshold'}\n")

    print(line + "\nHOW THIS SCORE WAS CALCULATED\n" + line)
    AV, AC, PR, UI, iss, imp, ex = cvss_parts(r["av"], r["ac"], r["pr"], r["ui"], r["scope"], r["c"], r["i"], r["a"])
    print("STEP 1  CVSS base score (one of the model's inputs, from the CVSS v3.1 formula)")
    print(f"        exploitability = 8.22 x AV({AV}) x AC({AC}) x PR({PR}) x UI({UI}) = {ex:.2f}")
    print(f"        impact         = {imp:.2f}   (from C/I/A, scope {r['scope']})")
    print(f"        base score     = round-up(min(impact + exploitability, 10)) = {r['cvss']:.1f}\n")

    nz = x.iloc[0][x.iloc[0] != 0]
    act = pd.DataFrame({"value": nz, "forest_importance": pd.Series(rf.feature_importances_, index=X.columns)[nz.index]})
    print(f"STEP 2  Turn the CVE into {x.shape[1]} numeric features ({len(nz)} are non-zero); the most influential active ones:")
    for name, rr in act.sort_values("forest_importance", ascending=False).head(8).iterrows():
        print(f"        {name:<26} value = {rr['value']:<8.3g} importance in forest = {rr['forest_importance']:.3f}")
    print()

    print(f"STEP 3  Random forest: {len(tree_p)} decision trees each read these features and output a probability")
    bins = np.histogram(tree_p, bins=[0, .05, .1, .2, .3, .5, 1.0001])[0]
    for lab, cnt in zip(["0-5%", "5-10%", "10-20%", "20-30%", "30-50%", "50%+"], bins):
        print(f"        trees saying {lab:<7}: {cnt:>3}  {'#' * int(round(cnt / len(tree_p) * 40))}")
    print(f"        raw forest score = average of the {len(tree_p)} trees = {p_raw:.3f}")
    print("        (raw scores are inflated on purpose by the class weights, so they are NOT yet real probabilities)\n")

    ys, xs = rf_iso.y_thresholds_, rf_iso.X_thresholds_
    same = xs[np.isclose(ys, p)]
    lo, hi = (same.min(), same.max()) if len(same) else (p_raw, p_raw)
    print("STEP 4  Calibration (isotonic regression fitted on the validation period)")
    print(f"        raw scores between {lo:.2f} and {hi:.2f} were exploited {p:.1%} of the time in validation data")
    print(f"        so   raw {p_raw:.3f}  ->  calibrated probability {p:.1%}\n")

    c_miss, c_alarm = CFG["cost_fn"] * p, CFG["cost_fp"] * (1 - p)
    print("STEP 5  Cost-sensitive decision (a missed exploit is 50x worse than a false alarm)")
    print(f"        expected cost if we IGNORE it : {CFG['cost_fn']} x {p:.3f}       = {c_miss:.3f}")
    print(f"        expected cost if we PATCH it  : {CFG['cost_fp']} x {1 - p:.3f}   = {c_alarm:.3f}")
    print(f"        threshold learned on validation data = {THRESH:.1%}  ->  {'patch it' if p >= THRESH else 'leave it'}\n")

    rank_pct = float((all_cal >= p).mean() * 100)
    print(f"STEP 6  Rank: this probability puts it in the top {rank_pct:.1f}% of all {len(df):,} CVEs\n")

    contrib = []
    for g, cols in FEATURE_GROUPS.items():
        xo = pd.concat([x] * len(BG), ignore_index=True);  xo[cols] = BG[cols].to_numpy()
        contrib.append((g, p_raw - float(rf.predict_proba(xo)[:, 1].mean())))
    contrib.sort(key=lambda t: -abs(t[1]));  mx = max(abs(d) for _, d in contrib) or 1.0
    print(f"STEP 7  What drove the score (a typical CVE scores {base_raw:.2f} raw; this one {p_raw:.2f})")
    print("        each group is swapped for values from 80 real CVEs and the change in raw score is measured:")
    for g, d in contrib:
        print(f"        {g:<30} {d:+.3f}  {('+' if d >= 0 else '-') * int(round(abs(d) / mx * 24))}")
    if known:
        print(f"\nHINDSIGHT (known only for dataset CVEs): " +
              (f"added to KEV on {pd.Timestamp(r['kev_date']).date()}." if pd.notna(r["kev_date"]) else "never added to KEV."))
    print("=" * 78)

In [ ]:
def ask(prompt, default):
    try:
        s = input(f"{prompt} [{default}]: ").strip()
    except Exception:                      # non-interactive run (no keyboard) -> use the default
        s = ""
    return s or default


best = scored[scored.split == "test"].sort_values("p_exploit", ascending=False)["cve_id"].tolist()
print("Example CVE ids from the held-out test period:")
print("   highest scored :", ", ".join(best[:4]))
print("   lowest scored  :", ", ".join(best[-3:]))
print("Type one of them, or type NEW to describe your own CVE.\n")

choice = ask("CVE id (or NEW)", best[0]).upper()
if choice == "NEW":
    print("\nDescribe the CVE (press Enter to accept the value in brackets):")
    fields = dict(
        product=ask("product (e.g. log4j, apache_http, chrome, zlib, openssl)", "log4j"),
        av=ask("attack vector  N=network A=adjacent L=local P=physical", "N").upper(),
        ac=ask("attack complexity  L=low H=high", "L").upper(),
        pr=ask("privileges required  N=none L=low H=high", "N").upper(),
        ui=ask("user interaction  N=none R=required", "N").upper(),
        scope=ask("scope  U=unchanged C=changed", "C").upper(),
        c=ask("confidentiality impact  H/L/N", "H").upper(), i=ask("integrity impact  H/L/N", "H").upper(),
        a=ask("availability impact  H/L/N", "H").upper(),
        cwe=int(ask("CWE number (e.g. 502, 79, 89, 787)", "502")), n_refs=int(ask("number of references", "12")),
        description=ask("description", "allows attackers to cause remote code execution via a crafted packet"),
    )
    explain_cve(None, **fields)
else:
    explain_cve(choice)